# Deconstructing the Tool Architecture, class-based

### Secure Agentic AI Systems (COMP9098), Week 3 in-class example

A standalone worked example for this week's lecture, Agent Architectures and Tool Use. This version defines a tool by subclassing `smolagents`' `Tool` class, spelling out the five components the framework needs (name, description, inputs, output type, forward) instead of leaving them implicit.

## Setup, run once

In [ ]:
!pip install -q -U huggingface_hub smolagents

In [ ]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo
from smolagents import CodeAgent, InferenceClientModel, Tool

# --- Get a Hugging Face token without ever writing it into the notebook file ---
# If you're in Colab and have added HF_TOKEN under the key (🔑) icon in the left sidebar, this picks it up automatically.
# Otherwise it will ask you to paste your token (starts with 'hf_'). This is only kept in memory for this session.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass

if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face token (hf_...): ")

assert isinstance(HF_TOKEN, str) and HF_TOKEN.startswith("hf_"), (
    "HF_TOKEN doesn't look like a Hugging Face token (should be a string starting with 'hf_'). "
    "Re-check the Colab secret value, or re-paste it when prompted."
)

## 1. Define the tool (explicit class-based architecture)

By subclassing `Tool`, we explicitly map out the 5 core components. This is exactly how the framework translates our code into something the LLM can call.

In [ ]:
class TimezoneTool(Tool):
    # Component 1: NAME
    name = "get_time_in_timezone"

    # Component 2: DESCRIPTION (The "When to use this" prompt)
    description = "Fetches the exact current local time for a given timezone."

    # Component 3: INPUTS (The JSON Schema for arguments)
    inputs = {
        "timezone_string": {
            "type": "string",
            "description": "A valid timezone name, like 'Europe/London' or 'Asia/Tokyo'. Use a full IANA identifier."
        }
    }

    # Component 4: OUTPUTS (What data type is returned as the Observation)
    output_type = "string"

    # Component 5: FORWARD (The execution logic that runs locally)
    def forward(self, timezone_string: str) -> str:
        print(f"\n[LOCAL EXECUTION] LLM paused. Local machine is calculating time for: {timezone_string}...")

        try:
            tz = ZoneInfo(timezone_string)
            current_time = datetime.now(tz)
            formatted_time = current_time.strftime("%I:%M %p on %A")

            print(f"[LOCAL EXECUTION] Calculation complete: {formatted_time}. Waking up LLM...\n")
            return f"The time in {timezone_string} is {formatted_time}."
        except Exception as e:
            return f"Error: Could not find timezone {timezone_string}. Please check the spelling."

## 2. Initialize the LLM and agent

In [ ]:
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    provider="auto",
    token=HF_TOKEN
)

# Instantiate our explicit tool class
my_timezone_tool = TimezoneTool()

timezone_agent = CodeAgent(
    tools=[my_timezone_tool],  # Pass the instantiated tool
    model=model,
    add_base_tools=False
)

## 3. Run the agent (observe the loop)

In [ ]:
print("Starting the Agent Loop...\n")

query = """
I am currently in Europe/London. I want to call my colleague who lives in Asia/Tokyo.
Can you check the time in Asia/Tokyo right now and tell me if it is a polite time to call them
(assuming they sleep between 11:00 PM and 7:00 AM)? Please use a full IANA timezone identifier such as
Asia/Tokyo or Europe/London when calling the tool.
"""

result = timezone_agent.run(query)

print("\n========================================")
print("FINAL AGENT OUTPUT (Loop Complete):")
print("========================================")
print(result)